Домашнее задание 5. Question answering

В ноутбуке решаются только основные пункты 1-5. Дополнительные задания с дообучением T5Gemma и LLM-as-a-judge здесь не выполняются. Ноутбук рассчитан на Kaggle GPU P100 и использует небольшие модели.

In [ ]:
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q sentence-transformers --no-deps
!pip install -q transformers tokenizers huggingface-hub safetensors tqdm scikit-learn scipy pillow
!pip install -q rank_bm25 ir_measures
!pip install -q bert-score

In [1]:
!nvidia-smi


Sun May  3 15:08:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.listdir('/kaggle/input/datasets/smakov/search-dataset/search-dataset')

['documents.csv', 'mirage', 'wikIR1k']

In [3]:
import os, re, json, math, time, random, gc, string
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

RNG_SEED = 42
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

WIKI_DIR   = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/wikIR1k'
DOCS_CSV   = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/documents.csv'
MIRAGE_DIR = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/mirage'

EVAL_LIMIT = 1000
QA_BATCH_SIZE = 32
CE_BATCH_SIZE = 64
GEN_BATCH_SIZE = 8
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 32

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2026-05-03 15:08:21.865446: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777820902.072177    1660 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777820902.133616    1660 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777820902.619084    1660 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777820902.619161    1660 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777820902.619164    1660 computation_placer.cc:177] computation placer alr

device: cuda


### Данные

In [4]:
with open(f'{MIRAGE_DIR}/dataset.json', encoding='utf-8') as f:
    mirage_dataset = json.load(f)
with open(f'{MIRAGE_DIR}/doc_pool.json', encoding='utf-8') as f:
    mirage_pool = json.load(f)
with open(f'{MIRAGE_DIR}/oracle.json', encoding='utf-8') as f:
    mirage_oracle = json.load(f)

oracle_chunk_by_qid = {k: v['doc_chunk'] for k, v in mirage_oracle.items()}
answers_by_qid = {item['query_id']: item['answer'] if isinstance(item['answer'], list) else [item['answer']] for item in mirage_dataset}

pool_idx_by_qid = defaultdict(list)
for i, ch in enumerate(mirage_pool):
    pool_idx_by_qid[ch['mapped_id']].append(i)

random.seed(RNG_SEED)
dataset_by_src = defaultdict(list)
for d in mirage_dataset:
    dataset_by_src[d['source']].append(d)

train_queries_m, test_queries_m = [], []
for src, qs in dataset_by_src.items():
    idx = list(range(len(qs)))
    random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_queries_m.extend(qs[i] for i in idx[:cut])
    test_queries_m.extend(qs[i] for i in idx[cut:])

eval_examples = test_queries_m if EVAL_LIMIT is None else test_queries_m[:EVAL_LIMIT]

print(f'MIRAGE: всего {len(mirage_dataset)} запросов, pool {len(mirage_pool)} чанков')
print(f'train: {len(train_queries_m)}, test: {len(test_queries_m)}, eval: {len(eval_examples)}')
print('sources in eval:', Counter(x['source'] for x in eval_examples))

MIRAGE: всего 7560 запросов, pool 37800 чанков
train: 6047, test: 1513, eval: 1000
sources in eval: Counter({'popqa': 615, 'naturalqa': 203, 'triviaqa': 117, 'ifqa': 50, 'drop': 15})


### Метрики

Для каждого эксперимента считаются EM и F1 в стиле SQuAD, EM-loose и EM-strict в стиле MIRAGE, а также BERTScore

MIRAGE EM-loose равен 1, если любая правильная строка ответа входит в предсказание как подстрока после lower case. MIRAGE EM-strict равен 1, если предсказание полностью совпадает с одним из ответов после lower case

In [5]:
def squad_normalize(text):
    text = str(text).lower()
    text = ''.join(ch if ch not in string.punctuation else ' ' for ch in text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in {'a', 'an', 'the'}]
    return ' '.join(tokens)

def squad_em(pred, refs):
    p = squad_normalize(pred)
    return float(any(p == squad_normalize(r) for r in refs))

def squad_f1(pred, refs):
    pred_tokens = squad_normalize(pred).split()
    best = 0.0
    for ref in refs:
        ref_tokens = squad_normalize(ref).split()
        common = Counter(pred_tokens) & Counter(ref_tokens)
        num_same = sum(common.values())
        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            score = float(pred_tokens == ref_tokens)
        elif num_same == 0:
            score = 0.0
        else:
            precision = num_same / len(pred_tokens)
            recall = num_same / len(ref_tokens)
            score = 2 * precision * recall / (precision + recall)
        best = max(best, score)
    return best

def mirage_em_loose(pred, refs):
    p = str(pred).lower()
    return float(any(str(r).lower() in p for r in refs))

def mirage_em_strict(pred, refs):
    p = str(pred).lower().strip()
    return float(any(str(r).lower().strip() == p for r in refs))

def clean_answer(text):
    text = str(text).strip()
    text = re.sub(r'^(answer|short answer|final answer)\s*:\s*', '', text, flags=re.I)
    text = text.split('\n')[0].strip()
    text = text.strip().strip(string.punctuation)
    return text

def basic_metrics(pred_by_qid):
    rows = []
    for item in eval_examples:
        qid = item['query_id']
        pred = clean_answer(pred_by_qid.get(qid, ''))
        refs = answers_by_qid[qid]
        rows.append({
            'qid': qid,
            'source': item['source'],
            'question': item['query'],
            'prediction': pred,
            'answers': refs,
            'squad_em': squad_em(pred, refs),
            'squad_f1': squad_f1(pred, refs),
            'mirage_em_loose': mirage_em_loose(pred, refs),
            'mirage_em_strict': mirage_em_strict(pred, refs),
        })
    return pd.DataFrame(rows)

all_predictions = {}
all_details = {}


### Ранжирование

В четвертом домашнем задании на MIRAGE лучшим был pretrained cross-encoder MiniLM. Вторым сильным основным вариантом была смешанная модель BM25 и all-mpnet-base-v2 с alpha 0.10

In [9]:
from sentence_transformers import CrossEncoder

CE_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
ce_model = CrossEncoder(CE_MODEL_NAME, device=DEVICE, max_length=512)

pairs, owners, cand_positions = [], [], []
for item in eval_examples:
    qid = item['query_id']
    for pos in pool_idx_by_qid[qid]:
        pairs.append((item['query'], mirage_pool[pos]['doc_chunk']))
        owners.append(qid)
        cand_positions.append(pos)

ce_scores = ce_model.predict(pairs, batch_size=CE_BATCH_SIZE, show_progress_bar=True)
ce_rankings = defaultdict(list)
for qid, pos, score in zip(owners, cand_positions, ce_scores):
    ce_rankings[qid].append((pos, float(score)))
ce_rankings = {qid: [pos for pos, score in sorted(vals, key=lambda x: -x[1])] for qid, vals in ce_rankings.items()}

del ce_model, pairs, owners, cand_positions, ce_scores
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('CE rankings:', len(ce_rankings))


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

CE rankings: 1000


In [10]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def tokenize(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())

chunk_texts = [ch['doc_chunk'] for ch in mirage_pool]
chunk_tokens = [tokenize(text) for text in chunk_texts]
bm25_mirage = BM25Okapi(chunk_tokens)

BI_MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
bi_model = SentenceTransformer(BI_MODEL_NAME, device=DEVICE)

chunk_emb = bi_model.encode(
    chunk_texts, batch_size=256, show_progress_bar=True,
    convert_to_tensor=True, normalize_embeddings=True, device=DEVICE
)
query_texts = [item['query'] for item in eval_examples]
query_ids = [item['query_id'] for item in eval_examples]
query_emb = bi_model.encode(
    query_texts, batch_size=128, show_progress_bar=True,
    convert_to_tensor=True, normalize_embeddings=True, device=DEVICE
)

ALPHA = 0.10
hybrid_rankings = {}
for item, qe in tqdm(list(zip(eval_examples, query_emb)), desc='hybrid rank'):
    qid = item['query_id']
    cand = pool_idx_by_qid[qid]
    all_bm25_scores = bm25_mirage.get_scores(tokenize(item['query']))
    bm25_scores = np.asarray([all_bm25_scores[i] for i in cand], dtype=np.float32)
    bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)
    cos_scores = (chunk_emb[cand] @ qe).detach().cpu().numpy()
    cos_norm = (cos_scores + 1.0) / 2.0
    score = ALPHA * bm25_norm + (1.0 - ALPHA) * cos_norm
    order = np.argsort(-score)
    hybrid_rankings[qid] = [cand[i] for i in order]

del bi_model, chunk_emb, query_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Hybrid rankings:', len(hybrid_rankings))


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/148 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

hybrid rank:   0%|          | 0/1000 [00:00<?, ?it/s]

Hybrid rankings: 1000


In [11]:
def truncate_context(text, max_chars=6000):
    text = str(text)
    return text if len(text) <= max_chars else text[:max_chars]

def oracle_context(item):
    return truncate_context(oracle_chunk_by_qid[item['query_id']])

def ranked_context(item, rankings, k):
    qid = item['query_id']
    positions = rankings[qid][:k]
    text = '\n\n'.join(mirage_pool[pos]['doc_chunk'] for pos in positions)
    return truncate_context(text)

def ce_top1_context(item):
    return ranked_context(item, ce_rankings, 1)

def hybrid_top1_context(item):
    return ranked_context(item, hybrid_rankings, 1)

def ce_top5_context(item):
    return ranked_context(item, ce_rankings, 5)

def hybrid_top5_context(item):
    return ranked_context(item, hybrid_rankings, 5)

### Извлечение ответов

Для пункта 2 используется encoder модель distilbert-base-cased-distilled-squad. 

Она дообучена на SQuAD и применима к oracle passage. Для пункта 4 та же модель применяется к top-1 passage от двух лучших ранжировщиков из четвертого домашнего задания

In [12]:
from transformers import pipeline

QA_MODEL_NAME = 'distilbert/distilbert-base-cased-distilled-squad'
qa_pipe = pipeline(
    'question-answering',
    model=QA_MODEL_NAME,
    tokenizer=QA_MODEL_NAME,
    device=0 if DEVICE == 'cuda' else -1,
)

def run_extractive(name, context_fn):
    pred_by_qid = {}
    for start in tqdm(range(0, len(eval_examples), QA_BATCH_SIZE), desc=name):
        batch_items = eval_examples[start:start + QA_BATCH_SIZE]
        questions = [item['query'] for item in batch_items]
        contexts = [(context_fn(item) or ' ') for item in batch_items]
        outputs = qa_pipe(question=questions, context=contexts, batch_size=QA_BATCH_SIZE, max_seq_len=384, doc_stride=128)
        if isinstance(outputs, dict):
            outputs = [outputs]
        for item, out in zip(batch_items, outputs):
            pred_by_qid[item['query_id']] = clean_answer(out.get('answer', ''))
    all_predictions[name] = pred_by_qid
    return pred_by_qid

run_extractive('2_extractive_oracle_distilbert_squad', oracle_context)
run_extractive('4_extractive_ce_top1', ce_top1_context)
run_extractive('4_extractive_hybrid_top1', hybrid_top1_context)

del qa_pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


2_extractive_oracle_distilbert_squad:   0%|          | 0/32 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


4_extractive_ce_top1:   0%|          | 0/32 [00:00<?, ?it/s]

4_extractive_hybrid_top1:   0%|          | 0/32 [00:00<?, ?it/s]

### LLM

Для генерации используется Qwen2.5-0.5B-Instruct. Модель помещается на P100. 

* Пункт 1 выполняется без контекста
* Пункт 3 выполняется с oracle passage
* Пункт 4 выполняется с top-1 passage от двух ранжировщиков
* Пункт 5 выполняется только для своих top-5 passages от двух ранжировщиков из четвертого домашнего задания

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM


SLM_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
slm_model = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True,
).to(DEVICE)
slm_model.eval()
print('loaded:', SLM_MODEL_NAME)


loaded: Qwen/Qwen2.5-0.5B-Instruct


In [26]:
def make_prompt(question, context=None):
    if context is None:
        user = (
            'Answer the question with a short phrase only. Do not explain.\n'
            f'Question: {question}\n'
            'Short answer:'
        )
    else:
        user = (
            'Use the context to answer the question. Answer with a short phrase only. Do not explain.\n'
            f'Context:\n{context}\n\n'
            f'Question: {question}\n'
            'Short answer:'
        )
    messages = [
        {'role': 'system', 'content': 'You are a concise question answering system.'},
        {'role': 'user', 'content': user},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return user

@torch.inference_mode()
def run_slm(name, context_fn=None):
    pred_by_qid = {}
    for start in tqdm(range(0, len(eval_examples), GEN_BATCH_SIZE), desc=name):
        batch_items = eval_examples[start:start + GEN_BATCH_SIZE]
        prompts = []
        for item in batch_items:
            context = None if context_fn is None else context_fn(item)
            prompts.append(make_prompt(item['query'], context))
        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(DEVICE)
        outputs = slm_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated = outputs[:, inputs['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for item, text in zip(batch_items, texts):
            pred_by_qid[item['query_id']] = clean_answer(text)
        del inputs, outputs, generated
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    all_predictions[name] = pred_by_qid
    return pred_by_qid

run_slm('1_slm_closed_book_zero_shot', None)
run_slm('3_slm_oracle_open_book', oracle_context)
run_slm('4_slm_ce_top1', ce_top1_context)
run_slm('4_slm_hybrid_top1', hybrid_top1_context)
run_slm('5_slm_ce_top5', ce_top5_context)
run_slm('5_slm_hybrid_top5', hybrid_top5_context)

5_slm_ce_top5:   0%|          | 0/125 [00:00<?, ?it/s]

5_slm_hybrid_top5:   0%|          | 0/125 [00:00<?, ?it/s]

{'069d0822-9164-4d38-8000-2c2a2296a647': 'Reha Erdem',
 '544304d6-4328-4bce-bd41-9ff0921dbddf': 'Lon Chaney',
 '3114a2a8-057f-4b91-8d15-767332950e3d': 'Haruki Kadokawa',
 '06b1d919-c967-4d36-8e66-aa5b5db10316': 'Richard Wallace',
 'e7151810-c3f9-416e-9613-7e5c17bba502': 'Sydney',
 'cd863204-19ef-4ad6-86e8-531b29ca2f73': 'Frank Borzage',
 '709803ef-2bfd-4f35-936e-9889cab6e731': 'Mario Bonnard',
 '80ee7c93-2be4-46a4-9a03-071f1fb9964a': 'John Heyman',
 '4b0b142e-1254-423d-bd92-6fc0b4419371': '1945). Bordagaray was selected by the White Sox in the 1934 draft. He was signed by the White Sox in the',
 '32fb79c2-19ba-4739-89c5-de99611a3415': 'PJ Hogan',
 'f44872a7-2c6e-4000-ad0c-6730eb68fe56': 'James Cameron',
 '5c49a99e-0ac1-4c38-a449-14afeaade3db': 'John Marco Allegro',
 'e6f698ac-90eb-4f43-821f-c440ede8db1c': 'Lynne Littman',
 '5ec81d02-9865-4987-aa15-c6c58b005e2e': 'Kim Mills',
 '58dc5635-5221-4996-b2cb-fa2938b92f41': 'Larry David',
 'fce97fdd-c14e-4720-90ca-e763638ed362': 'Shi Hu',
 '4f1

### Оценка

Ниже для всех полученных ответов считаются метрики. Для BERTScore используется distilbert-base-uncased 

Если у вопроса несколько правильных ответов, берется максимальный BERTScore по вариантам ответа

In [27]:
from bert_score import BERTScorer

BERTSCORE_MODEL = 'distilbert-base-uncased'
bertscorer = BERTScorer(
    model_type=BERTSCORE_MODEL,
    device=DEVICE,
    batch_size=64,
    rescale_with_baseline=False,
)

def add_bertscore(df):
    pair_preds, pair_refs, owners = [], [], []
    for i, row in df.iterrows():
        refs = row['answers']
        if not refs:
            refs = ['']
        for ref in refs:
            pair_preds.append(row['prediction'] if row['prediction'] else ' ')
            pair_refs.append(str(ref) if str(ref) else ' ')
            owners.append(i)
    _, _, f1 = bertscorer.score(pair_preds, pair_refs, verbose=True)
    f1 = f1.detach().cpu().numpy()
    best = np.full(len(df), -np.inf, dtype=np.float32)
    for owner, value in zip(owners, f1):
        best[owner] = max(best[owner], float(value))
    return best

summary_rows = []
for name, preds in all_predictions.items():
    df = basic_metrics(preds)
    df['bertscore_f1'] = add_bertscore(df)
    all_details[name] = df
    summary_rows.append({
        'experiment': name,
        'SQuAD_EM': df['squad_em'].mean(),
        'SQuAD_F1': df['squad_f1'].mean(),
        'MIRAGE_EM_loose': df['mirage_em_loose'].mean(),
        'MIRAGE_EM_strict': df['mirage_em_strict'].mean(),
        'BERTScore_F1': df['bertscore_f1'].mean(),
    })

summary = pd.DataFrame(summary_rows).sort_values('experiment').reset_index(drop=True)
summary_rounded = summary.copy()
for col in summary_rounded.columns[1:]:
    summary_rounded[col] = summary_rounded[col].map(lambda x: round(float(x), 4))
print(summary_rounded.to_string(index=False))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

done in 1.64 seconds, 1717.15 sentences/sec
calculating scores...
computing bert embedding.


  0%|          | 0/48 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/45 [00:00<?, ?it/s]

done in 1.68 seconds, 1676.02 sentences/sec
                          experiment  SQuAD_EM  SQuAD_F1  MIRAGE_EM_loose  MIRAGE_EM_strict  BERTScore_F1
         1_slm_closed_book_zero_shot     0.008    0.0528            0.033             0.008        0.6768
2_extractive_oracle_distilbert_squad     0.623    0.7399            0.742             0.610        0.9166
              3_slm_oracle_open_book     0.672    0.7671            0.743             0.658        0.9249
                4_extractive_ce_top1     0.538    0.6481            0.645             0.527        0.8878
            4_extractive_hybrid_top1     0.439    0.5360            0.522             0.429        0.8522
                       4_slm_ce_top1     0.590    0.6799            0.655             0.576        0.8975
                   4_slm_hybrid_top1     0.476    0.5641            0.538             0.465        0.8622
                       5_slm_ce_top5     0.512    0.5948            0.569             0.506        0.8674
  

### Итоговый анализ

Оценка проводилась на 1000 вопросах MIRAGE test

Без контекста маленькая языковая модель почти не справляется: `1_slm_closed_book_zero_shot` дает только `SQuAD_EM = 0.008` и `SQuAD_F1 = 0.0528`. 

Лучший результат получился с oracle passage: `3_slm_oracle_open_book` достигает `SQuAD_EM = 0.672`, `SQuAD_F1 = 0.7671`, `BERTScore_F1 = 0.9249`. Extractive QA на oracle context тоже сильный, но немного уступает SLM по строгим метрикам.

Среди retrieved context лучше работает cross-encoder. Для SLM top-1 CE дает `SQuAD_EM = 0.590`, а hybrid BM25 + mpnet — `0.476`. Такая же тенденция видна и для extractive QA, поэтому результаты подтверждают вывод из четвертого домашнего задания о преимуществе cross-encoder на MIRAGE

Top-5 context не улучшил качество: CE top-5 снизился до `SQuAD_EM = 0.512`. Вероятно, дополнительные passages добавляют шум и частично обрезаются по длине контекста. В этой конфигурации для маленькой SLM лучше использовать один самый релевантный passage, а не пять